# PhotoGraphiQ manuscript experiment suite (R1-R60)

Generated by `build_notebook.py` from the experiment scripts; do not edit by hand. Each section shows the script's docstring (question, model, reference, acceptance criteria), runs the unmodified script in a subprocess and displays the figures it wrote. Results, provenance and captions are written by the scripts themselves under `results/`, `figures/` and `figures/captions/`.

Select what runs with `TIERS` in the setup cell: `{"smoke"}` takes minutes, `{"smoke", "safe"}` is the default light run, and adding `"heavy"` with `FULL = True` reproduces the publication configuration (hours; the timing studies should run on an otherwise idle machine). A missing optional dependency is reported as skipped, never as passed. See `EXPERIMENT_INDEX.md` and `STATUS.md`.

## Install dependencies (once per environment)

In [ ]:
# Core, development and autodiff extras from the local checkout (builds the native engines).
import subprocess
import sys

INSTALL = False  # set True on a fresh environment
if INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "..[dev,autodiff]"], check=True)


## Setup

In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path

from IPython.display import Image, Markdown, display

PAPER_EXPERIMENTS_DIR = Path.cwd()
if not (PAPER_EXPERIMENTS_DIR / "run_all_safe.py").exists():
    PAPER_EXPERIMENTS_DIR = Path("paper_experiments").resolve()
sys.path.insert(0, str(PAPER_EXPERIMENTS_DIR))
import run_all_safe  # noqa: E402  (the single source of tiers and optional dependencies)

TIERS = {"smoke", "safe"}  # add "heavy" (and set FULL = True) for the publication run
FULL = False
ENTRIES = {run_all_safe.experiment_id(e[0]): e for e in run_all_safe.SCRIPTS}
RECORD = {}


def run_experiment(rid):
    script, tier, needs, reason, accepts_full = ENTRIES[rid]
    import importlib.util

    missing = [m for m in needs if importlib.util.find_spec(m) is None]
    if tier not in TIERS:
        RECORD[rid] = "skipped"
        display(Markdown(f"*Skipped: tier `{tier}` not selected ({reason or 'see TIERS'}).*"))
        return
    if missing:
        RECORD[rid] = "skipped"
        display(Markdown(f"*Skipped: optional dependency missing: {', '.join(missing)}.*"))
        return
    path = PAPER_EXPERIMENTS_DIR / script
    command = [sys.executable, str(path)] + (["--full"] if FULL and accepts_full else [])
    start = time.perf_counter()
    done = subprocess.run(command, cwd=path.parent, capture_output=True, text=True)
    RECORD[rid] = "passed" if done.returncode == 0 else "FAILED"
    print(f"{rid}: {RECORD[rid]} in {time.perf_counter() - start:.1f} s")
    print(done.stdout[-3000:])
    if done.returncode != 0:
        print(done.stderr[-3000:])
        raise RuntimeError(f"{rid} failed its acceptance criteria; see the output above")
    for figure in sorted((PAPER_EXPERIMENTS_DIR / "figures" / "png").glob(f"{rid}_*.png")):
        display(Image(filename=str(figure), width=760))


## R1: Basic CV wire / one-step teleportation.

Script: `01_core_mbqc/01_wire_teleportation.py`

Verifies the elementary CV-MBQC transport primitive: a single p-homodyne
measurement on a two-mode momentum-squeezed resource teleports a Fourier
transform of the input mode (``photographiq.protocols.wire`` with one shear).

Exact functionality: the unconditional affine channel (S, N, d) is computed
independently by ``photographiq.analysis.gaussian_channel``, which propagates
Wigner variables through the pattern and is a separate code path from the
conditional trajectory simulator (``simulate``/``run_shots``). Agreement
between the two is evidence that finite-squeezing ensemble simulation matches
the analytic finite-resource channel derived in docs/theory.md.

Finite-resource approximation: resource squeezing r is finite (r=1.0 here);
noise diag(0, exp(-2r)) is physical, not a truncation artifact.

Upgrade migration (2026-11): the ensemble is now sampled on three Gaussian
trajectory engines -- NumPy ``gaussian``, the C++ ``native-gaussian`` and the
Piquasso-backed ``piquasso`` (kept as an external reference). In addition to
the original fixed tolerance, every first and second moment of each ensemble
is checked against the analytic channel within 5 shot-noise standard errors,
the corrected output channel (S, N) is checked against an independently
written Fourier map F and noise diag(0, e^{-2r}), and the native engine is
checked against the NumPy engine trajectory by trajectory (documented RNG
contract: same Generator calls, equal seeds give equal trajectories up to
floating-point summation order).

In [ ]:
run_experiment("R1")

## R2: Identity wire versus finite squeezing.

Script: `01_core_mbqc/02_identity_finite_squeezing.py`

Sweeps resource squeezing r for the package's actual four-step identity wire
(``photographiq.protocols.identity``, i.e. four Fourier teleportation steps
whose ideal composed map is identity) and measures how the output mean error
and added covariance noise shrink as r grows.

Exact functionality: ``gaussian_channel`` computes the unconditional affine
map (S, N, d) analytically for the fixed-angle pattern; per docs/theory.md an
L-step wire accumulates noise recursively and four k=0 steps implement
identity with noise 2*exp(-2r)*I. That closed form is reproduced here
independently (via ``photographiq.gaussian.wire_channel``, a separate
function from the pattern-level analyzer) as a second oracle.

Upgrade migration (2026-11): at each r the pattern is also EXECUTED on the
NumPy ``gaussian`` and C++ ``native-gaussian`` trajectory engines
(``run_shots``, same seeds). The sampled ensemble mean/covariance must agree
with the analytic channel within 5 shot-noise standard errors, and the two
engines must agree trajectory by trajectory (documented RNG contract). The
identity map and the added noise are reported separately: the noise is the
physical cost of finite squeezing, never a compilation error.

In [ ]:
run_experiment("R2")

## R3: Rotation compiler validation.

Script: `02_compiler/03_rotation_compile.py`

For each rotation angle theta, compiles ``Circuit(1).rotate(0, theta)`` to a
finite-squeezing MBQC pattern, extracts its exact unconditional Gaussian
channel, and compares the resulting linear map against an independently
written NumPy rotation matrix (not ``photographiq.gaussian.rotation``, which
is the function the compiler itself is built from).

Exact functionality: in the r -> infinity limit the compiled channel matrix
converges to the ideal rotation; finite r contributes physical additive noise
N, not a truncation artifact. A second sweep fixes theta and varies resource
squeezing to show the noise floor separately from the angle dependence.

Upgrade migration (2026-11): representative compiled patterns are also
EXECUTED with ``run_shots`` on NumPy ``gaussian`` and C++ ``native-gaussian``
(see ``execute_compiled``). The sampled ensembles must match the analytic
channel (map AND added noise) within 5 shot-noise standard errors and the two
engines must agree trajectory by trajectory. The map error (compilation) and
the added noise N (finite squeezing) stay in separate columns.

In [ ]:
run_experiment("R3")

## R4: Squeezing compiler validation.

Script: `02_compiler/04_squeezing_compile.py`

Compiles ``Circuit(1).squeeze(0, r_logical)`` for a sweep of logical squeezing
parameters, extracts the exact unconditional Gaussian channel and compares it
to an independently written NumPy squeezing matrix diag(exp(-r), exp(r))
(``photographiq.gaussian.squeezing`` uses the identical formula but is the
function the compiler is built from, so an inline copy is used as the
independent oracle here). Numerical stability is monitored via the condition
number of the compiled map and the growth of the added noise as |r| grows.

Upgrade migration (2026-11): three representative compiled patterns are also
EXECUTED on NumPy ``gaussian`` and C++ ``native-gaussian`` (``execute_compiled``):
sampled ensembles vs the analytic channel within 5 shot-noise standard
errors, native vs NumPy trajectory by trajectory. Map error and added noise
stay in separate columns.

In [ ]:
run_experiment("R4")

## R5: Random single-mode symplectic compilation (major validation result).

Script: `02_compiler/05_random_symplectic_compile.py`

Draws many random valid single-mode Gaussian transformations
S = R(phi1) S(r) R(phi2) with a deterministic RNG, compiles each through
``Circuit(1).gaussian(0, S).compile(...)``, and compares the compiled affine
map against the exact target S using the independent Gaussian-channel
analyzer. ``decompose_symplectic`` can raise for numerically unstable near-
singular requests (docs/theory.md); those cases are recorded as decomposition
failures rather than silently skipped.

Upgrade migration (2026-11): every compiled pattern's graph metadata is
validated against its command list and the typed IR ``OpenGraph``
(``graph_metadata_checks``); the first EXEC_SAMPLES compiled patterns are
EXECUTED on NumPy ``gaussian`` and C++ ``native-gaussian`` (``execute_compiled``)
and compared with the analytic channel (map and added noise) within 5
shot-noise standard errors. The added noise N is reported in its own column
(noise_trace) and is never counted as compilation error.

In [ ]:
run_experiment("R5")

## R6: Two-mode CZ compilation.

Script: `02_compiler/06_cz_compile.py`

Sweeps the logical controlled-Z weight in ``Circuit(2).cz(0, 1, weight)``,
compiles each circuit, and compares the exact unconditional two-mode Gaussian
channel to an independently written analytic CZ symplectic matrix
CZ(g): p_i -> p_i + g*q_j (photographiq.gaussian.cz implements the identical
convention; the copy here is written independently as the cross-check
target). For representative coherent inputs, mean/covariance and the
generated cross-mode correlations are also compared.

Upgrade migration (2026-11): representative compiled patterns are also
EXECUTED on NumPy ``gaussian`` and C++ ``native-gaussian`` (``execute_compiled``):
sampled ensembles vs the analytic channel (map and added noise) within 5
shot-noise standard errors; native vs NumPy trajectory by trajectory. The
added noise is reported in its own column, never as compilation error.

In [ ]:
run_experiment("R6")

## R7: Beamsplitter compilation.

Script: `02_compiler/07_beamsplitter_compile.py`

Sweeps the beamsplitter mixing angle theta in ``Circuit(2).beamsplitter``,
compiles the (deliberately resource-expensive, per docs/theory.md) SUM-gate
decomposition, and compares the exact unconditional Gaussian channel to an
independently written analytic beamsplitter symplectic matrix (the package's
own convention: c=cos, s=sin on (q0,p0,q1,p1)). Records the MBQC resource
overhead (nodes/measurements/commands), which the docs flag as especially
costly for this gate.

Upgrade migration (2026-11): representative compiled patterns are also
EXECUTED on NumPy ``gaussian`` and C++ ``native-gaussian`` (``execute_compiled``):
sampled ensembles vs the analytic channel (map and added noise) within 5
shot-noise standard errors; native vs NumPy trajectory by trajectory. The
added noise is reported in its own column, never as compilation error.

In [ ]:
run_experiment("R7")

## R8: Multi-mode circuit-to-MBQC compilation showcase (main-text figure candidate).

Script: `02_compiler/08_multimode_showcase.py`

Builds a five-mode Gaussian circuit mixing rotations, a squeezer, a CZ gate,
a beamsplitter and a displacement, compiles it with an explicit provenance
trace, and renders the package's own gate-to-resource visualization
(``photographiq.visualize_compilation``). Also records the compiled
resource-graph statistics (nodes, measurements, entanglement edges,
corrections, commands) that accompany the figure.

Beamsplitter compilation is deliberately resource-expensive (SUM-gate
decomposition via three CZ/rotation sandwiches per docs/theory.md), so the
circuit here is kept small enough to stay under
``visualize_compilation``'s built-in size guard (<=24 gates, <=150 nodes).

Upgrade migration (2026-11):

* the compiled 10x10 symplectic map and displacement are checked against an
  independently written NumPy product of the eight gates (the added
  finite-squeezing noise N is reported separately, never as map error);
* the graph metadata is validated against the command list and the typed IR
  ``OpenGraph`` (edges/weights, node squeezing, each non-output node measured
  once, feed-forward after the measurement it reads, IR round trip);
* the pattern is EXECUTED on NumPy ``gaussian`` and C++ ``native-gaussian``
  and the sampled ensemble is compared with the analytic channel;
* the main figure is a measurement-order view: x = position of the node's
  homodyne in the command order, y = logical mode, colour = source gate,
  red arrows = classical feed-forward (measurement -> later correction). The
  package's spring-layout overview is kept as ``R8_multimode_showcase_package_view``.

In [ ]:
run_experiment("R8")

## R9: Compilation provenance / trace.

Script: `02_compiler/09_compilation_trace.py`

Uses the same five-mode showcase circuit as R8 and exports the full
gate-by-gate compilation provenance (``CompilationTrace``/``CompilationStep``)
as a machine-readable CSV plus a human-readable text summary.

Upgrade migration (2026-11): the trace is now also checked against the
command list it describes: every index in ``step.measurements`` is a
``Measure`` command, every index in ``step.corrections`` is a ``Displace``
command, every feed-forward dependency of a correction points to an earlier
command (``Pattern.dependencies``), and the trace's resource nodes are
exactly the prepared (non-input) nodes. No backend is involved; R8 executes
the same compiled pattern on ``gaussian`` and ``native-gaussian``.

In [ ]:
run_experiment("R9")

## R10: Compiler resource scaling.

Script: `02_compiler/10_compiler_scaling.py`

Builds circuits with an increasing number of repeated gates (rotation,
squeezing, CZ, beamsplitter) and records how compiled resource-node count,
measurement count, entanglement-edge count, command count and compile time
grow. Beamsplitter compilation is a fixed, angle-independent SUM-gate
decomposition that is markedly more resource-expensive per gate (see R7), so
its gate-count sweep is capped lower to keep total runtime practical.

Upgrade migration (2026-11): the LARGEST compiled pattern of every gate kind
is additionally (a) checked against the independently composed ideal map
(rotations/squeezers/CZs/beamsplitters with summed parameters commute within
each kind), with the added noise reported separately, and (b) executed once
on NumPy ``gaussian`` and C++ ``native-gaussian`` with the same seed, which
must give the same homodyne records (documented RNG contract). Compile times
are wall-clock observations on a shared machine (provisional).

In [ ]:
run_experiment("R10")

## R11: Adaptive feed-forward.

Script: `03_adaptive_flow/11_adaptive_feedforward.py`

Builds a three-node line resource where the *measurement angle* of the
second homodyne detector depends on the first measurement's outcome
(``pg.Outcome("m0")``), in addition to the standard outcome-dependent
displacement corrections. This is genuinely adaptive: the affine
``gaussian_channel`` analyzer explicitly rejects outcome-dependent gate
angles (docs/theory.md), which is used here as a structural check that the
pattern is not secretly representable as a fixed-angle Gaussian channel.
Reproducibility is achieved by fixing the trajectory seed.

Upgrade migration (2026-11): the historical docstring said Gaussian homodyne
had no explicit-outcome API; ``simulate(..., measurement_outcomes=...)`` now
postselects homodyne records on the Gaussian engines. The script therefore
also (1) runs every seed on NumPy ``gaussian`` and C++ ``native-gaussian``
(records must agree, documented RNG contract), (2) evaluates a grid of
FIXED outcomes (m0, m1) on both engines and compares each conditional output
with a non-adaptive control pattern whose second angle is set by hand to
0.05*m0 + theta -- equality proves the angle was resolved from the earlier
outcome, (3) checks that different m0 give different conditional outputs (the
feed-forward is not inert) and (4) checks the measurement order in
``Pattern.schedule()``. Postselected runs are conditional branches, not
deterministic execution.

In [ ]:
run_experiment("R11")

## R12: Dependency DAG / causal schedule.

Script: `03_adaptive_flow/12_dependency_dag.py`

Builds a four-node adaptive pattern with multiple classical dependencies
(two homodyne outcomes feeding three downstream corrections/measurements),
extracts ``Pattern.dependencies()`` (a command DAG) and
``Pattern.schedule()`` (a topological command ordering), and checks that
every dependency edge points strictly backward-to-forward in that schedule.

Upgrade migration (2026-11): the schedule is also EXECUTED. The commands are
re-assembled in ``Pattern.schedule()`` order into a new pattern, which must
validate and must reproduce the original pattern's seeded records and output
moments on NumPy ``gaussian`` and C++ ``native-gaussian`` for several seeds
(different outcome branches). Records of the two engines must agree
(documented RNG contract).

In [ ]:
run_experiment("R12")

## R13: Invalid causality / software-safety test (appendix result).

Script: `03_adaptive_flow/13_invalid_causality.py`

Constructs several intentionally invalid patterns and confirms that
``Pattern.validate()`` (or construction itself) rejects each one with the
expected, actual validator error — not a fabricated exception. This audits
the software's causal/structural safety checks rather than its physics.

Upgrade migration (2026-11): every invalid pattern is additionally handed
straight to ``simulate`` on the C++ ``native-gaussian`` engine without a prior
``validate()`` call; it must be rejected with the same exception type and
message before any state is prepared (``simulate`` validates first). A second
table uses ``photographiq.flows`` on CV open graphs: a graph with no CV flow
(rank-deficient correction matrix) must be classified ``impossible`` with
exact rational witnesses, and a qubit flow algorithm applied to a CV graph
must return ``unsupported`` -- never a qubit certificate for a CV graph.

In [ ]:
run_experiment("R13")

## R14: CV-flow validation.

Script: `03_adaptive_flow/14_cvflow.py`

Uses the current, supplied-total-order CV-flow certificate
(``photographiq.cvflow.certify_cv_flow``/``flow_pattern``), which implements
the real-linear condition of Booth and Markham (Quantum 7, 1146, 2023) for
one given measurement order. It does NOT search over orders, so this script
only demonstrates certification/rejection for two concrete orders on the
same three-node line resource -- not a general optimal-flow finder.

Valid example: measuring the line graph 0->1->2 (input 0, output 2) in the
natural order (0, 1) yields a zero-residual certificate. Invalid example:
measuring the same graph in the reversed order (1, 0) has no exact real-linear
correction (node 1 alone cannot supply the needed influence on the surviving
output once node 0, its only other neighbor, is excluded as an input), so
certification returns None with reported residuals in the diagnostic case.

Upgrade migration (2026-11): the historical script predates the CV-flow
FINDER in ``photographiq.flows``. ``flows_study`` now adds:

* ``flows.analyze`` on the same line graph (typed IR ``OpenGraph``,
  ``Model.CV_GAUSSIAN``): status, decision, regimes, measurement order and the
  correction map, which must equal the supplied-order certificate above;
* the corrected pattern built from exactly that correction map
  (``flows.cv.corrected_pattern``), executed on NumPy ``gaussian`` and C++
  ``native-gaussian`` over several sampled outcomes (records must agree), with
  the ensemble compared with ``gaussian_channel``;
* a deliberately WRONG correction map (q-correction scaled by 0.5), which
  ``verify_cv_determinism`` must reject;
* cases where the algorithm must DECLINE certification: a singular bridge
  (``impossible`` with exact rational witnesses), a near-singular bridge
  (det 1e-10: ``unknown``/``numerically_uncertain`` by default, ideal-limit
  certificate only with ``exact=True``), a unit wire at r = 4 (``unknown`` with
  the default float64 check shift 3.0, certified with shift 1.5), a qubit
  algorithm on a CV graph and a non-Gaussian model (``unsupported``).

A qubit flow certificate is never used for a CV graph, and ``not_certified``
or ``unknown`` is never read as proof of nondeterminism except where an
exact impossibility witness is returned.

In [ ]:
run_experiment("R14")

## R15: Raw Piquasso Gaussian reference (independent, not through PhotoGraphiQ).

Script: `04_piquasso_validation/15_raw_piquasso_gaussian.py`

For displacement, rotation, squeezing, beamsplitter, CZ (native
``pq.ControlledZ``, the correct Gaussian-regime representation; Piquasso's
``GaussianTransform`` is instead used by PhotoGraphiQ's *Fock* backend) and a
short compound circuit, this script:

1. Builds a raw ``piquasso`` program directly with the Piquasso public API
   (no PhotoGraphiQ wrapper call in the numerical path) starting from vacuum,
   preparing the same coherent input and applying the same native gate.
2. Runs the identical physical operation through PhotoGraphiQ's
   ``backend="piquasso"`` path.
3. Compares mean-vector norm error, covariance Frobenius error and mean
   photon number (via the documented analytic Gaussian formula, applied
   identically to both sides) across many random parameter draws.

Upgrade migration (2026-11): step 2 now runs the identical pattern on all
three PhotoGraphiQ Gaussian engines -- ``piquasso`` (historical columns, names
unchanged), NumPy ``gaussian`` and C++ ``native-gaussian`` -- and every engine
must match raw Piquasso to the same tolerance. Raw Piquasso stays the
independent reference (no PhotoGraphiQ code in its numerical path).

In [ ]:
run_experiment("R15")

## R16: Independent NumPy analytic Gaussian reference.

Script: `04_piquasso_validation/16_numpy_analytic_reference.py`

Builds small symplectic target matrices directly with NumPy (not
``photographiq.gaussian``), applies them analytically to a coherent-state
input, and compares the result against three execution paths: PhotoGraphiQ's
NumPy Gaussian backend, PhotoGraphiQ's Piquasso Gaussian backend, and a raw
Piquasso program built independently of PhotoGraphiQ's backend adapters.

Upgrade migration (2026-11): the C++ ``native-gaussian`` engine is added to
the exact Gaussian comparison (same 1e-9 tolerance). The three Fock engines
``native-fock``, ``piquasso-fock`` and ``native-mixed-fock`` run the same
patterns at a fixed total-photon cutoff; their per-mode quadrature means and
variances are compared with the analytic target. That is a truncated-basis
approximation, so it has its own declared tolerance (FOCK_TOLERANCE) and the
minimum retained Fock norm is reported next to it -- truncation is never
hidden by renormalisation.

In [ ]:
run_experiment("R16")

## R17: Graphix structural comparison (structure only, never CV amplitudes).

Script: `05_graphix_validation/17_graphix_structure.py`

For a line and a star resource topology, builds a hand-matched Graphix
qubit-MBQC ``Pattern`` (N/E/M/X/Z commands) alongside the equivalent
PhotoGraphiQ CV pattern, and compares only *structural* invariants: resource
graph isomorphism/edges, input/output node sets, measurement order, and
classical-dependency domains. Continuous-variable amplitudes, quadrature
statistics or any numerical physics are never compared against Graphix,
per the package's documented validation policy (docs/validation/index.md).

Upgrade migration (2026-11): a second, still purely structural comparison
uses the upgraded flow layer. For each resource graph the QUBIT open graph
(``photographiq.ir.OpenGraph``, ``Model.QUBIT``, XY measurements) is analysed
with ``photographiq.flows.analyze(og, "causal")`` and with Graphix's own
causal-flow finder (``graphix.opengraph.OpenGraph.to_causalflow_or_none``);
existence and the X-correction support of every measured node must agree.
The same graph typed as ``Model.CV_GAUSSIAN`` must be rejected by the qubit
algorithm (``unsupported``): Graphix and qubit flow are never used as a CV
simulator or as a CV determinism certificate.

In [ ]:
run_experiment("R17")

## R18: Gaussian backend cross-check.

Script: `06_backend_crosschecks/18_gaussian_backends.py`

Runs the same random, low-energy, purely physical (non-adaptive) two-mode
Gaussian workloads through the NumPy Gaussian backend and the Piquasso
Gaussian backend (both exact) and compares state mean/covariance/photon
number/quadrature exactly. It additionally runs the same preparation through
the experimental ``piquasso-mixed-fock`` backend at a modest cutoff and
compares photon number/quadrature -- this third comparison is a *finite Fock
truncation approximation*, not an exact-backend agreement, and uses a looser,
explicitly reported tolerance.

Upgrade migration (2026-11): the C++ ``native-gaussian`` engine joins the
exact comparison (same 1e-8 tolerance against NumPy ``gaussian``). The
finite-cutoff comparison now covers all four Fock engines --
``native-fock``, ``piquasso-fock``, ``native-mixed-fock`` and the historical
``piquasso-mixed-fock`` (columns ``mixed_fock_*`` keep their old meaning) --
at the same cutoff, reporting photon-number and quadrature-mean errors
against the exact Gaussian state and the minimum retained Fock norm of each
run. Engines that hit the norm guard are recorded, never silently dropped.

In [ ]:
run_experiment("R18")

## R19: Photon subtraction analytic probability.

Script: `07_non_gaussian/19_photon_subtraction_probability.py`

Uses the package's physical vacuum-tap subtraction pattern
(``photographiq.non_gaussian.photon_subtraction``): a beamsplitter of angle
theta couples input |n> to a vacuum ancilla, which is then photon-counted.
For input |n>, heralding count k has the exact binomial probability
C(n,k) sin(theta)^(2k) cos(theta)^(2(n-k)) -- this is the analytic reference,
derived independently here with SciPy, and the resulting signal-mode photon
number is deterministically n-k (a number-conserving beamsplitter on a
definite Fock input produces a photon-number-correlated joint state).

Upgrade migration (2026-11): the main engine is now the C++ ``native-fock``
backend (columns ``photographiq_*``, ``conditional_photon_number``). The
historical engine ``piquasso-fock`` and the density-matrix engine
``native-mixed-fock`` run the identical fixed-outcome branches as
independent references; every engine must match the analytic binomial law
and the conditional photon number n-k to the original 1e-9 tolerance. The
minimum retained Fock norm of each run is recorded (no truncation is hidden).

In [ ]:
run_experiment("R19")

## R20: Photon subtraction on squeezed vacuum.

Script: `07_non_gaussian/20_subtracted_squeezed_state.py`

Prepares a squeezed-vacuum input (``photographiq.states.GaussianInput.squeezed``,
which the pure-Fock backend accepts directly and converts to a truncated
Fock vector) and performs one-photon heralded subtraction via
``photographiq.non_gaussian.photon_subtraction``. Compares the Fock
distribution, parity and mean photon number before/after, and demonstrates
the expected even-to-odd parity flip (squeezed vacuum has only even photon
numbers; single-photon subtraction produces a cat-like odd-photon-dominant
state). Uses ``cutoff_convergence`` to check the finite-cutoff behavior of
this filtering/attenuating physical subtraction (not an ideal a-operator).

Upgrade migration (2026-11): the main engine is the C++ ``native-fock``
backend. The same conditional branch (count = 1) is recomputed on
``piquasso-fock`` (historical engine) and ``native-mixed-fock`` as independent
references: herald probability, parity, mean photon number and the
total-variation distance of the photon-number distribution must agree within
REFERENCE_TOLERANCE. The cutoff-convergence study runs on ``native-fock`` and
its retained norms are reported; the state is never silently renormalised
without the retained norm being recorded. The minimum of the Wigner function
(negativity) is reported for the subtracted state.

In [ ]:
run_experiment("R20")

## R21: Cat state.

Script: `07_non_gaussian/21_cat_state.py`

Generates even and odd finite-Fock cat resources (``photographiq.resources.
CatResource``, which internally calls ``FockInput.cat``) for several alpha
values, and cross-checks against an independently written NumPy coherent-
state expansion (not reusing ``FockInput.cat``). Reports parity,
photon-number distribution, normalization/retained mass and Wigner functions,
and checks the expected even/odd parity sign.

Upgrade migration (2026-11): the main engine is the C++ ``native-fock``
backend; ``piquasso-fock`` (historical) and ``native-mixed-fock`` prepare the
same resources as references. Every engine must reach fidelity
>= 1 - 1e-6 with the independent NumPy expansion and the correct parity sign;
the engines' parities and mean photon numbers are compared with each other.
Retained norm and boundary population are reported for each engine.

In [ ]:
run_experiment("R21")

## R22: Cubic phase direct execution.

Script: `07_non_gaussian/22_cubic_phase_direct.py`

Applies ``photographiq.commands.CubicPhase(gamma)`` (native Piquasso
``exp(i*gamma*q^3/6)``, hbar=2) to a low-energy coherent input on the pure
Fock backend, and compares the resulting truncated state vector against an
independently built finite-dimensional reference: a dense ladder operator
``a`` constructed directly with NumPy, ``q = a + a^T``, and
``U = scipy.linalg.expm(i*gamma*q^3/6)`` applied to the SAME truncated input
vector. This is a finite-Fock-truncation comparison at matched cutoff, not an
infinite-cutoff convergence claim (see R33/R34 for that).

Upgrade migration (2026-11): main engine ``native-fock`` with its default
cubic-phase definition ``truncated_generator`` (expm of the truncated
generator, the same finite-dimensional operator as the reference, so the
fidelity must be 1 to rounding); ``piquasso-fock`` is kept as the historical
reference with the same criterion. The script previously had no acceptance
test; FIDELITY_TOLERANCE is now declared and asserted. The native
``cubic_phase="projected"`` definition (exact projected matrix elements,
reports leakage above the cutoff) is a DIFFERENT finite-cutoff operator: its
fidelity to the truncated-generator reference and its retained norm are
reported, not asserted, and converge as the cutoff grows (see R34).

In [ ]:
run_experiment("R22")

## R23: Cubic MBQC injection (finite-resource approximation -- clearly labelled).

Script: `07_non_gaussian/23_cubic_injection.py`

Uses ``photographiq.non_gaussian.cubic_injection(gamma, squeezing=r)`` with a
FIXED ancilla homodyne outcome m=0 (so the nonlinear feed-forward corrections
QuadraticPhase(-2*gamma*m) and Displace(p=-gamma*m^2) vanish identically,
isolating the residual finite-resource Gaussian envelope). The result is
compared against an ideal direct CubicPhase(gamma) applied to the same
truncated input, using the same independent NumPy/SciPy matrix-exponential
reference as R22. Per docs/non_gaussian.md this injection is a conditional
filter times a cubic phase, NOT an exact finite-energy unitary cubic gate;
fidelity to the ideal gate is expected to improve as resource squeezing r
grows (the envelope widens toward the infinite-squeezing/no-filter limit).

Upgrade migration (2026-11): main engine ``native-fock``; ``piquasso-fock``
(historical) runs the same grid as reference. The two engines treat the
two-mode CZ differently at a finite cutoff (native: exact projected Gaussian
matrix elements; Piquasso: Euler decomposition truncated after each factor),
so cells that hit the retained-norm guard can differ between engines; every
failed cell is recorded with its error, never dropped. Where both engines
succeed, the fidelity between their output states is reported. Asserted
(declared before evaluation): at the largest cutoff the native-fock fidelity
to the ideal gate does not decrease with resource squeezing among successful
cells (tolerance MONOTONE_SLACK), and native/Piquasso outputs agree to
ENGINE_FIDELITY_FLOOR wherever both succeed. The m = 0 branch is a
postselected conditional result, not deterministic execution.

In [ ]:
run_experiment("R23")

## R24: Kerr direct execution.

Script: `07_non_gaussian/24_kerr_direct.py`

Applies ``photographiq.commands.Kerr(kappa)`` (native Piquasso
``exp(i*kappa*n^2)``) to the explicit custom superposition
``(|0>+|1>+|2>)/sqrt(3)`` (``FockInput`` with arbitrary amplitudes) and
compares each output amplitude against the exact analytic phase rule
``|n> -> exp(i*kappa*n^2)|n>``, computed independently with NumPy.

Upgrade migration (2026-11): main engine ``native-fock`` (Kerr is an exact
diagonal phase there); ``piquasso-fock`` (historical) and the density-matrix
engine ``native-mixed-fock`` are run with the same criteria. For the mixed
engine the fidelity is <e|rho|e> and the "state-vector error" column holds the
Frobenius distance ||rho - |e><e|||.

In [ ]:
run_experiment("R24")

## R25: Kerr synthesis convergence (approximate; synthesis-only, no MBQC noise).

Script: `07_non_gaussian/25_kerr_synthesis.py`

``photographiq.synthesis.synthesize_kerr`` returns an ordinary ``Circuit``
of native rotation/quadratic-phase/cubic-phase gates approximating
exp(i*kappa*n^2) by a Lie product formula. To isolate the product-formula
approximation error from separate finite-resource MBQC injection error
(covered by R23), this script translates the synthesized circuit's gates
directly into PHYSICAL Fock-backend commands (Rotate/QuadraticPhase/
CubicPhase/Displace) and executes them without compiling to an MBQC pattern.
Fidelity to the exact phase rule is compared across synthesis_steps.

Upgrade migration (2026-11): every step count is executed on ``native-fock``
(main) and ``piquasso-fock`` (historical reference). The two engines realise
QuadraticPhase differently at a finite cutoff (exact projected matrix
elements vs Piquasso's truncated Euler decomposition), so their fidelities
are reported side by side and their mutual output fidelity is recorded.
Declared check: on each engine the infidelity at the largest step count is
smaller than at one step (the Lie-product error decreases). Runtimes are
provisional (shared machine).

In [ ]:
run_experiment("R25")

## R26: GKP resource characterization.

Script: `08_gkp/26_gkp_resource.py`

Generates finite-energy square-lattice GKP resources
(``photographiq.gkp.GKPResource``), characterizes their q-quadrature
probability structure (comparing the intended smooth target wavefunction to
its finite-Fock projection, reconstructed from the Hermite basis functions
the package itself uses -- ``photographiq.fock_measurements.wavefunctions``),
stabilizer expectations (``photographiq.gkp.stabilizers``), logical-codeword
overlap (``GKPCode.gram``), and normalization/cutoff sensitivity via
``GKPResource.project``. This makes NO fault-tolerance or threshold claim.

Upgrade migration (2026-11): the resources are loaded on ``native-fock``
(main) and ``piquasso-fock`` (historical reference); both engines must give
the same stabilizer expectations and mean photon number to
ENGINE_TOLERANCE. The cutoff-sensitivity sweep now also evaluates the
stabilizers on ``native-fock`` at every cutoff (cutoff requirement of the
stabilizer estimate itself), next to the captured mass of the projection.
The projection's captured mass is always reported: the Fock input is the
normalised projection, so 1 - captured_mass is the discarded weight.

In [ ]:
run_experiment("R26")

## R27: GKP logical shifts/stabilizers.

Script: `08_gkp/27_gkp_stabilizers.py`

Applies logical lattice displacements (``photographiq.gkp.logical_displacement``,
X: q -> q+sqrt(2*pi), Z: p -> p+sqrt(2*pi)) to finite-energy GKP codewords and
checks the resulting stabilizer expectations and decoded bit against the
ideal square-lattice relations, computed independently with NumPy. Finite
energy means these are approximate, not exact, permutations of the codeword.

Upgrade migration (2026-11): the codewords and shifted codewords are now
evolved on ``native-fock`` (main) and ``piquasso-fock`` (historical
reference); the engines must agree on every stabilizer modulus to
ENGINE_TOLERANCE. New logical observable: the single-shot logical READOUT
ERROR PROBABILITY of q-homodyne + nearest-cell decoding, computed exactly
(no sampling) by integrating the q-marginal |psi(q)|^2 of each Fock state
over the cells that decode to the wrong bit. The expected bit is L for the
codeword and for Z(L), and 1 - L after the logical X shift. The integrated
density mass is reported (it must be ~1, i.e. the grid captures the state).
Declared checks: engines agree; every error probability is below 1/2. These
are properties of finite-energy codewords under ideal homodyne detection --
not logical error rates of a fault-tolerant scheme.

In [ ]:
run_experiment("R27")

## R28: GKP syndrome extraction (finite-energy/Fock approximation).

Script: `08_gkp/28_gkp_syndrome.py`

Prepares a finite-energy GKP |0>_L codeword, physically displaces it by a
known small q-shift, then runs the package's actual one-ancilla SUM syndrome
extraction (``photographiq.gkp.correction_pattern``) with a finite "logical
plus" ancilla. A single trajectory's raw ancilla homodyne outcome samples one
of many comb peaks of the (also finite-energy) ancilla, so the decoded
residual has substantial single-shot noise set by the resource's intrinsic
peak width; this script instead reports the ENSEMBLE (circular) mean and
spread over repeated seeded trajectories, which is the physically meaningful
way to test whether extraction is centered on the true displacement. This is
a finite-ancilla, finite-energy approximation, not a fault-tolerant claim.

Upgrade migration (2026-11): the main engine is ``native-fock`` (two GKP
modes share one total-photon cutoff, dimension binom(81, 2) = 3240). For
REFERENCE_DELTAS the identical seeded trajectories are re-run on
``piquasso-fock`` (historical engine); the two engines share the Python
sampling path, so equal seeds must give the same syndrome records up to the
small numerical difference of the evolved states (REFERENCE_RECORD_TOLERANCE)
-- this is checked and reported. Failure rates are now reported with Wilson
95% intervals: (a) syndrome error rate, |decoded residual - expected
residual| (circular) > sqrt(2 pi)/4; (b) simulation failures (norm guard),
counted separately. (The mean |<q>| of the corrected state is NOT used as a
failure criterion: it is ~1.9 at every delta, identical to the historical
Piquasso result, and plausibly reflects the finite-envelope ancilla's partial
measurement of the data q rather than logical errors -- not verified here.)
All are
finite-energy, finite-ancilla single-round statistics, not logical error
rates of a fault-tolerant architecture.

In [ ]:
run_experiment("R28")

## R29: Finite squeezing noise scan (main-text result).

Script: `09_noise/29_finite_squeezing.py`

Fixes a compound logical circuit (single-mode rotate+squeeze, and a two-mode
CZ-entangled circuit) and sweeps the MBQC RESOURCE squeezing used to compile
it. For each resource squeezing value, extracts the exact unconditional
Gaussian channel (S, N, d) and reports: the logical-map error against an
independently built NumPy target matrix (expected to stay near machine
precision, since squeezing does not change the ideal map), Tr(N), the
spectral norm/max eigenvalue of the added-noise matrix N, and the resulting
output variance for a fixed coherent input.

Upgrade migration (2026-11): (1) the CZ circuit's added noise N is compared
with an independent closed form (four identity-wire steps per output mode,
``gaussian.wire_channel``, block diagonal; the instantaneous Entangle adds no
noise); (2) at EXEC_SQUEEZING the compiled patterns are EXECUTED on NumPy
``gaussian`` and C++ ``native-gaussian`` (``execute_compiled``): sampled
ensembles must reproduce the analytic channel -- ideal map AND added noise --
within 5 shot-noise standard errors, and the engines must agree trajectory by
trajectory. The logical-map error and the added noise stay separate columns:
finite squeezing is never reported as compilation error.

In [ ]:
run_experiment("R29")

## R30: Loss / attenuation validation.

Script: `09_noise/30_loss_validation.py`

Sweeps intensity transmissivity eta through ``photographiq.commands.Loss``
on both Gaussian backends. For a coherent input, checks the exact analytic
relations alpha_out = sqrt(eta)*alpha_in and <n>_out = eta*<n>_in (thermal
occupation held at zero), computed independently with NumPy.

Upgrade migration (2026-11): the C++ ``native-gaussian`` engine is added to
the exact comparison (same 1e-9 tolerance). The density-matrix engines
``native-mixed-fock`` and ``piquasso-mixed-fock`` apply the same pure-loss
channel (Kraus decomposition) at MIXED_CUTOFF; their photon number and q
variance are compared with the analytic values, their minimum retained norm
and wall time per run (provisional, shared machine) are recorded to
characterise accuracy and cost. Asserted with MIXED_TOLERANCE when the
retained norm is >= 1 - 1e-9 (coherent |alpha| = 0.72: tail above 20 photons
is negligible).

In [ ]:
run_experiment("R30")

## R31: Thermal noise validation.

Script: `09_noise/31_thermal_noise.py`

Sweeps transmissivity eta and thermal environment occupation nbar through
``photographiq.commands.Loss`` on both Gaussian backends, and compares the
resulting mean/covariance to the independent analytic thermal-attenuation
channel: mean -> sqrt(eta)*mean_in, V -> eta*V_in + (1-eta)*(2*nbar+1)*I
(statistical covariance, vacuum V=I), written directly with NumPy.

Upgrade migration (2026-11): the C++ ``native-gaussian`` engine joins the
exact comparison (same tolerance). On a subset (MIXED_ETAS x NBARS) the
density-matrix engines ``native-mixed-fock`` and ``piquasso-mixed-fock`` apply
the thermal-loss Kraus channel at MIXED_CUTOFF. Thermal occupation spreads the
state over many Fock levels, so their photon-number and q-variance errors are
a genuine truncation effect that grows with nbar: they are reported together
with the minimum retained norm and wall time (provisional), and asserted
(MIXED_TOLERANCE) only where the retained norm is >= 1 - 1e-9.

In [ ]:
run_experiment("R31")

## R32: Detector inefficiency / noisy homodyne.

Script: `09_noise/32_noisy_measurement.py`

Noisy homodyne (``Homodyne(efficiency, noise)``) is supported on both
Gaussian backends and experimentally on the mixed-Fock backend; the pure
Fock backend explicitly rejects it (feature matrix, docs/validation/
feature-matrix.md). This script verifies the documented added-variance
formula nu = (1-efficiency)/efficiency + noise (docs/theory.md,
"Destructive Gaussian measurements") against sampled outcome variance on the
NumPy Gaussian backend, then separately cross-checks a modest mixed-Fock case
against the exact Gaussian prediction (a finite-cutoff approximation).

Upgrade migration (2026-11):

* the statistically powered sweep also runs on C++ ``native-gaussian`` with the
  same seed; its samples must equal the NumPy samples (documented RNG
  contract) and pass the same 6-sigma check;
* the mixed-Fock cross-check now runs on ``native-mixed-fock`` (main) and
  ``piquasso-mixed-fock`` (historical). NEW deterministic check: the
  postselected outcome DENSITY of the noisy homodyne (``measurement_outcomes``)
  at a grid of outcomes is compared with the analytic Gaussian density
  N(2 Re alpha, 1 + nu) -- exact up to Fock truncation, no shot noise, and
  ~0.04 s per point;
* sampling the noisy branch on a mixed-Fock engine costs ~8-22 s PER SHOT
  (``quad_vec`` inside a root find, shared Python path, so the native engine
  does not remove it; ISSUES_FOUND.md). The default configuration therefore
  samples only the noiseless cases (MIXED_FOCK_SHOTS_NOISELESS shots, 6-sigma
  checked); ``--full`` restores the historical noisy sampling
  (MIXED_FOCK_SHOTS_NOISY shots per noisy case per engine) as the reproducible
  larger configuration.

In [ ]:
run_experiment("R32")

## R33: Fock cutoff convergence.

Script: `10_convergence/33_fock_cutoff.py`

Runs representative non-Gaussian workloads (cat state, photon subtraction,
cubic phase, GKP resource) at strictly increasing Fock cutoffs using
``photographiq.convergence.cutoff_convergence``, and reports retained norm,
boundary population, photon number, adjacent-cutoff fidelity/trace distance
(from the study itself) plus fidelity to the HIGHEST tested cutoff (computed
separately here). The highest tested cutoff is a reference point, not an
exact/infinite-cutoff ground truth -- convergence must still be judged from
the trend, not assumed from the last value's smallness alone.

Upgrade migration (2026-11):

* every original workload runs on ``native-fock`` (main) and ``piquasso-fock``
  (historical engine); the fidelity between the two engines' states at each
  cutoff is reported and must be >= 1 - ENGINE_INFIDELITY (declared below);
* two DEMANDING Gaussian workloads with an exact reference were added, because
  the original low-energy workloads converge already at the lowest cutoff:
  (a) ``displaced_squeezed_gates``: vacuum -> Squeeze(0.5) -> Displace(q = 8,
  i.e. alpha = 4), mean photon number ~16, cutoffs 40..120, on ``native-fock``
  and ``piquasso-fock``; (b) ``thermal_displaced_squeezed``: a mixed Gaussian
  input (thermal 0.2, r = 0.3, alpha = 3) on ``native-mixed-fock`` and
  ``piquasso-mixed-fock`` -- this is the combined squeeze+displace Gaussian
  unitary whose native implementation was re-derived in this tree
  (docs/validation/native-fock-gaussian-unitary.md). Their photon number and
  <q> are compared with the exact Gaussian moments at every cutoff. Asserted:
  the native engines reach relative photon-number error <= NATIVE_EXACT_TOLERANCE
  at the highest cutoff. Piquasso's values are REPORTED (Piquasso 8.0.1's
  displacement/squeezing matrices are documented to lose accuracy at large
  cutoffs; see backends/native_fock.py), not asserted.

In [ ]:
run_experiment("R33")

## R34: Cubic injection convergence -- 2D scan over cutoff and resource squeezing.

Script: `10_convergence/34_cubic_convergence.py`

Two-dimensional grid of (Fock cutoff, resource squeezing) for
``photographiq.non_gaussian.cubic_injection`` with a fixed ancilla outcome
m=0 (isolating the residual finite-resource Gaussian envelope from outcome-
dependent feed-forward). Fidelity to an ideal direct CubicPhase(gamma),
computed independently with NumPy/SciPy, is saved as a heatmap-ready CSV.

Upgrade migration (2026-11): the grid runs on ``native-fock`` (main) and
``piquasso-fock`` (historical reference), each against the independent
reference built from its own truncated input. Cells that hit the retained-norm
guard (|norm - 1| > 1e-3 after an operation) are physical truncation of the
cubic-phase ancilla (its p spread grows like gamma q^2 with q ~ e^r), recorded
with the retained norm in the error message and shown as hatched cells, never
renormalised silently. Where both engines succeed their output fidelity must be
>= ENGINE_FIDELITY_FLOOR (truncation-level CZ differences only), and at the
largest cutoff the native fidelity to the ideal gate must not decrease with r
(MONOTONE_SLACK) -- both declared before evaluation. The m = 0 branch is
postselected, not deterministic.

In [ ]:
run_experiment("R34")

## R35: Fock-space dimension and memory model (analytical, cheap).

Script: `11_performance/35_fock_dimension_scaling.py`

Uses the actual PhotoGraphiQ total-photon Fock dimension formula
D = binom(modes+cutoff-1, modes) (``PiquassoFockBackend.dimension``, cross-
checked against ``common.fock_dimension``) over a modes x cutoff grid, and
estimates pure-vector and dense-density-matrix byte costs. The default
``max_dimension`` allocation guard is read directly from the backend class.
Dense-density-matrix bytes are a rough dense upper-style estimate; the actual
mixed-Fock implementation may differ (see docs/performance.md).

Upgrade migration (2026-11): the dimension formula of the C++ engine
(``NativeFockBackend.dimension``) is checked against the same independent
formula, and a MEASURED part was added: wall time of one two-mode workload
(two coherent inputs, BeamSplitter, Kerr, CubicPhase) on ``native-fock``,
``piquasso-fock`` and ``native-mixed-fock`` over a cutoff sweep, separated into
state preparation (inputs only) and gate execution (difference), with the
retained norm recorded and the engines' final states compared (fidelity).
``--quick`` uses a short sweep and few repeats to prove the script works;
the default is the publication sweep. Timings carry
``timing_status`` (provisional unless PGQ_IDLE_MACHINE=1 declares an idle
machine) and thread/build information.

In [ ]:
run_experiment("R35")

## R36: Gaussian runtime scaling.

Script: `11_performance/36_gaussian_runtime.py`

Benchmarks preparing and entangling an N-mode line resource
(``CVGraph.line(N)``) through the NumPy Gaussian backend and the Piquasso
Gaussian backend, with warm-up runs and repeated timing measurements
(median, IQR, std). Timing is a local observation of this machine, not a
comparative performance claim (docs/performance.md, docs/validation.md).

Upgrade migration (2026-11): the C++ ``native-gaussian`` engine is timed next
to NumPy ``gaussian`` and Piquasso ``piquasso``. Two workloads separate the
costs: ``prepare_entangle`` (state preparation + CZ entangling, the original
workload) and ``measure_feedforward`` (the same resource with every node but
the last q-homodyne measured and the next node displaced by the outcome);
their difference estimates the measurement + feed-forward cost. Before timing,
every engine's output is checked against NumPy ``gaussian`` (same seed: equal
records and moments, CORRECTNESS_TOLERANCE). ``--quick`` runs a short sweep
to prove the script; the default is the publication sweep. Timings carry
thread/build information and are provisional unless PGQ_IDLE_MACHINE=1.

In [ ]:
run_experiment("R36")

## R37: Compiler runtime scaling (with warm-up and repeated timing).

Script: `11_performance/37_compiler_runtime.py`

Benchmarks ``Circuit.compile`` alone (not execution) for a growing number of
alternating rotation/CZ logical gates on two modes, using warm-up runs and
repeated ``time.perf_counter`` measurements (median, IQR, std). Records
generated resource nodes, commands and measurements alongside timing.

Upgrade migration (2026-11): the cost of the pipeline is separated into
compilation (``Circuit.compile``), analytic channel extraction
(``gaussian_channel``) and execution of one trajectory on NumPy ``gaussian``
and C++ ``native-gaussian`` (same seed; records must agree, a correctness gate
before any timing is reported). The compiled map is checked against the
independently composed target (rotations about mode 0 and CZs, which do not
commute, are multiplied in order). ``--quick`` runs a short sweep to prove the
script; timings are provisional unless PGQ_IDLE_MACHINE=1.

In [ ]:
run_experiment("R37")

## R38: PhotoGraphiQ overhead versus raw Piquasso.

Script: `11_performance/38_piquasso_overhead.py`

Builds matched N-mode Gaussian workloads (per-mode squeeze+rotate, plus a CZ
chain) and times two independent execution paths with warm-up and repeated
measurements: (1) one single raw ``piquasso`` program executed once with one
``GaussianSimulator`` call (the natural, most efficient native-engine usage),
and (2) the identical physical operations through
``photographiq.simulate(..., backend="piquasso")``, which executes one native
Program/Simulator per gate (see ``backends/piquasso.py``). Raw Piquasso is
the numerical engine reference; PhotoGraphiQ adds graph/pattern/MBQC
abstractions on top, and the ratio T_PG/T_PQ quantifies that added cost on
this machine (not a comparative benchmark claim across machines).

Upgrade migration (2026-11): the same workload is also timed on NumPy
``gaussian`` and C++ ``native-gaussian`` (ratios to raw Piquasso reported as
additional columns; the original two columns are unchanged). A correctness
gate precedes timing: every engine's output covariance must match the raw
Piquasso state (CORRECTNESS_TOLERANCE). ``--quick`` runs a short sweep; the
default is the publication sweep; timings are provisional unless
PGQ_IDLE_MACHINE=1.

In [ ]:
run_experiment("R38")

## R39: Serialization round-trip.

Script: `12_serialization/39_serialization.py`

Serializes a nontrivial adaptive pattern (labelled nodes, ``Parameter`` and
``Outcome`` expression dependencies, an outcome-dependent measurement angle)
through the public versioned JSON API (``Pattern.to_json``/``Pattern.from_json``,
backed by ``photographiq.serialization``), and verifies the restored pattern
has structurally equal commands, preserved labels/expression dependencies,
and identical seeded execution behavior.

Upgrade migration (2026-11): the round trip is now checked field by field --
resource-graph edges and weights, node squeezing, ordered inputs/outputs,
measurement metadata (node, key, measurement type, efficiency, noise, angle
expression evaluated on sampled records), classical dependency DAG and the
correction functions (every ``Displace`` q/p expression evaluated on random
record values) -- and the restored pattern must give identical seeded results
on NumPy ``gaussian``, C++ ``native-gaussian`` and ``piquasso`` (backend-
independent semantics). The typed IR ``OpenGraph`` JSON round trip
(``to_dict``/``from_dict``) is checked on the same resource, and a
non-Gaussian pattern with nonlinear feed-forward
(``non_gaussian.cubic_injection``) is serialized, restored and executed on
``native-fock`` on a fixed outcome branch (identical states required).

In [ ]:
run_experiment("R39")

## R40: Seed reproducibility / shots.

Script: `12_serialization/40_seeded_shots.py`

Verifies that ``photographiq.run_shots`` gives bit-identical outcomes for
repeated runs with the same seed, and that two different seeds draw from
statistically indistinguishable distributions (a two-sample Kolmogorov-
Smirnov test) while producing different raw sample sequences.

Upgrade migration (2026-11): cross-engine reproducibility is tested where the
code DOCUMENTS an RNG contract, and only observed elsewhere:

* documented: ``native-gaussian`` draws outcomes in Python with exactly the
  same ``numpy.random.Generator`` calls as ``gaussian`` (module docstring of
  backends/native_gaussian.py: equal seeds give equal trajectories up to
  floating-point summation order), and ``simulate`` documents that
  ``piquasso`` and ``gaussian`` give identical seeded trajectories. Asserted:
  max |sample difference| <= CONTRACT_TOLERANCE for both pairs;
* observed only: ``native-fock`` and ``piquasso-fock`` share the Python
  sampling path of ``PiquassoFockBackend`` (no written cross-engine contract).
  Photon-number samples of a coherent state are compared and the fraction of
  equal outcomes is REPORTED, not asserted.

In [ ]:
run_experiment("R40")

## R41: JAX deterministic gradient validation (experimental; runs only if JAX is installed).

Script: `13_autodiff/41_jax_gradients.py`

For rotation, displacement, squeezing, cubic phase, Kerr and loss gates
(each with one JAX-differentiable ``Parameter``), compares
``jax.grad`` of ``photographiq.autodiff.expectation`` against central finite
differences of the SAME function at several step sizes h, per the package's
documented validation approach (docs/validation/feature-matrix.md:
"Estimator convergence: Four finite-difference steps"). This checks internal
autodiff/finite-difference consistency of the differentiable Fock generator,
not agreement with an external analytic formula.

Upgrade migration (2026-11): the native C++ engines have NO automatic
differentiation (they are not JAX-traceable; ``photographiq.autodiff`` builds
its own dense JAX generators and prepares inputs with the Piquasso-backed
``MixedFockBackend``). They are used here as an INDEPENDENT numerical
realisation: (1) the forward value of ``autodiff.expectation`` at t0 is
compared with the same observable computed by ``simulate`` on ``native-fock``
(``native-mixed-fock`` for the loss channel) at the same cutoff; (2) a central
FINITE DIFFERENCE of the native-engine observable (step NATIVE_FD_STEP) is
compared with the JAX gradient. Finite differences are labelled as such and
never reported as automatic differentiation. Declared tolerances:
FORWARD_TOLERANCE and NATIVE_FD_TOLERANCE (central-difference truncation
~h^2 times the third derivative plus rounding ~eps/h).

In [ ]:
run_experiment("R41")

## R42: Conditional / estimator gradients (experimental, statistical estimators).

Script: `13_autodiff/42_stochastic_gradients.py`

Tests the two generic Monte Carlo gradient-estimator primitives in
``photographiq.autodiff`` against known analytical identities:

1. ``score_function_surrogate`` (REINFORCE/likelihood-ratio estimator) on a
   parametrized Bernoulli branch, compared to the exact analytic gradient of
   E[f(X)] for X ~ Bernoulli(sigmoid(theta)).
2. ``gaussian_sample`` (pathwise/reparameterization primitive) on a simple
   quadratic objective, compared to the exact analytic pathwise gradient.

Both are Monte Carlo estimators: agreement is statistical (bias and standard
error over repeated seeds), not exact-precision agreement. This is
explicitly experimental functionality (docs/validation/feature-matrix.md).

Upgrade migration (2026-11): a third case applies the score-function
estimator to REAL homodyne records sampled by the C++ ``native-gaussian``
engine (``run_shots``): a coherent input displaced by Displace(q = theta) is
q-homodyned, f(m) = m^2, and the likelihood log p(m | theta) is the analytic
Gaussian density with mean 2 Re(alpha) + theta and unit variance (vacuum
noise, hbar = 2). Exact gradient: d/dtheta E[m^2] = 2 (2 Re(alpha) + theta).
The native engine supplies samples only; the gradient is taken by JAX on the
surrogate (automatic differentiation of the estimator, not of the simulator --
native engines have no autodiff). Same 5-standard-error criterion.

In [ ]:
run_experiment("R42")

## R43: Native Gaussian backend validation against analytic phase-space formulas.

Script: `14_native_gaussian/43_native_gaussian_validation.py`

Scientific question: does the compiled ``native-gaussian`` backend (in-place O(nk) C++
kernels) reproduce the exact Gaussian phase-space evolution (states, gates, loss,
homodyne/heterodyne conditioning, adaptive feed-forward) to floating-point accuracy, at
every mode count and circuit depth we use, and does it agree trajectory-by-trajectory with
the NumPy ``gaussian`` reference backend?

Model and conventions: hbar = 2, interleaved (q1, p1, ...), vacuum covariance I.

References (all independent of the code under test):

* Single-mode states: closed forms. Coherent mean (2 Re a, 2 Im a); squeezed covariance
  R(t) diag(e^{-2r}, e^{2r}) R(t)^T; thermal (2 nbar + 1) I; mean photon number
  (tr V - 2)/4 + |d|^2/4.
* Gates: Heisenberg map S = expm(2 Omega M) of each gate's quadratic generator
  U = exp(-i x^T M x / 2), embedded densely in 2n x 2n (``tests/validation/test_val_gaussian.py``
  helpers ``from_generator``/``gen_*``/``embed``, which are themselves checked against the
  Fock-space Heisenberg action in that test-suite). Loss: X = sqrt(eta) I,
  Y = (1-eta)(2 nth + 1) I.
* Conditioning: physical dilation (vacuum-ancilla beam splitter for detector inefficiency,
  50:50 splitter + two homodynes for heterodyne) and the PRECISION-matrix form of Gaussian
  conditioning (``_homodyne_reference``/``_heterodyne_reference`` of the same test module),
  not the Schur-complement formula the backends implement.
* Adaptive feed-forward: two-step teleportation wire with outcome-dependent measurement
  angle, propagated by hand (``_wire_step``).

Quantities: mean-vector error, covariance error (relative to max |V_ref|), symplectic
residual ||S Omega S^T - Omega||_max / ||S||_2^2 of the composed native map (S extracted
column by column from the native mean update), physicality min eig(V + i Omega), and
conditional-state / density agreement for the same outcomes.

Assumptions/limitations: double precision only; deep random circuits use bounded gate
parameters (|r| ~ 0.15) so that covariances stay O(10^2) and the relative-error criterion is
meaningful; the forward-error model of the acceptance bound is heuristic, not proven.

In [ ]:
run_experiment("R43")

## R44: Native Gaussian backend performance, measured together with accuracy.

Script: `14_native_gaussian/44_native_gaussian_performance.py`

Scientific question: what does the compiled ``native-gaussian`` backend (in-place O(n k)
local updates) buy over the NumPy ``gaussian`` backend (dense O(n^3) S V S^T per gate) and
over independent implementations (a dense NumPy symplectic propagation written here, and
Piquasso 8.0.1's GaussianSimulator executing the whole circuit as one program), as a function
of mode count, gate count, number of conditional measurements, batch size and OpenMP thread
count, and at what accuracy?

Model: seeded random circuits of rotations, squeezers, beam splitters, CZ, shears,
displacements and thermal loss (generator of R43) on a random mixed n-mode Gaussian state;
sequences of postselected homodyne measurements; batched symplectic updates and batched
homodyne conditioning (``photographiq.backends.native_gaussian.batch_*``) against a vectorised
NumPy dense update / Schur complement written here.

Measurement method: every configuration runs in a fresh subprocess (so peak memory is per
configuration and OMP_NUM_THREADS takes effect). Runtime: ``common.benchmark`` (warm-up
excluded, median and IQR of repeats). Memory: peak process working set from
``psutil.Process().memory_info().peak_wset`` (Windows; ``ru_maxrss`` elsewhere), reported as
absolute peak and as the increase over the peak just before the timed section (0 means the
run stayed below the import-time peak). tracemalloc is NOT used (it misses C++ allocations).
Gate timings exclude the O(n^3) physicality validation of state construction (an unvalidated
copy of a validated start state is loaded each repeat); the NumPy backend's own per-gate
validation is part of its cost and is included. Accuracy: max relative covariance error of
the final state against the dense first-principles reference of R43.

Threads: the native Gaussian single-trajectory kernels are serial; only the batched kernels
use OpenMP (cpp/src/gaussian/batch.cpp). OMP_NUM_THREADS is honoured (checked:
``_native.gaussian.max_threads`` follows it); it also limits OpenBLAS/MKL threads of NumPy in
the same process, which is stated with the results.

Timings are labelled final only when the run declares an idle machine (``PGQ_IDLE_MACHINE=1``);
otherwise they are marked provisional. The manuscript configuration is ``--full``.

In [ ]:
run_experiment("R44")

## R45: Native pure-state Fock backend accuracy, truncation and cutoff convergence.

Script: `15_native_fock/45_native_fock_accuracy.py`

Scientific question: how accurate is ``native-fock`` (compiled C++ engine, total-photon
truncation n_1 + ... + n_m < cutoff, gates applied as P U P with renormalisation and a
recorded retained norm) for Fock inputs, displacement, squeezing, beam splitters, Kerr,
cubic phase and conditional photon counting, and how do state, observable and norm errors
converge with the cutoff? Where does Piquasso 8.0.1 (piquasso-fock) agree or not?

Two kinds of reference are used and kept apart:

* exact-in-truncation references (what the engine SHOULD produce at this cutoff):
  P U |psi> / ||P U |psi>|| from closed forms (coherent and squeezed amplitudes with 30-digit
  mpmath, ``tests/validation/test_val_fock.py``), exact phases (Kerr), expm of the generator on
  an enlarged per-mode product space (beam splitter; ``unitary`` of the same test module), and the
  exact projected cubic-phase elements <m|exp(i gamma q^3/6)|n> by Hermite-function quadrature
  (``cubic_projected``). Their retained norm ||P U psi||^2 is also exact. Deviations here are
  rounding of the engine.
* untruncated physics (what the infinite-dimensional state gives): <n> = |alpha|^2, sinh^2 r;
  cubic phase on vacuum p -> p + gamma q^2 (hbar = 2, [q^3, p] = 6i q^2) gives <p> = gamma,
  <p^2> = 1 + 3 gamma^2;
  Kerr <a> from a 400-level sum; two-mode squeezed vacuum (two squeezers + 50:50 splitter)
  photon counting P(k) = (1 - l^2) l^(2k), l = tanh r, conditional state |k>. Deviations here
  are TRUNCATION, reported as convergence with the cutoff, never hidden.

Cubic phase: ``cubic_phase="projected"`` (exact P V P, leaks norm) and the default
``"truncated_generator"`` (expm of the truncated q^3, unitary on the truncated space, never
reports leakage) are both studied; only the former has an exact-in-truncation reference.

Assumptions/limitations: double precision; one- and two-mode systems (multimode scaling is
R46); the photon-counting check uses the exact factorisation P_tot (BS (S x S)) |00>, whose
conditional state is exactly |k> for 2k < cutoff.

In [ ]:
run_experiment("R45")

## R46: Fock-space optimisations that PhotoGraphiQ actually implements, with ablations.

Script: `15_native_fock/46_fock_space_optimization.py`

Scientific question: what do the implemented Fock-space optimisations buy in runtime and
memory, at identical physics and identical numerical output?

Implemented optimisations studied (inspected in cpp/src/fock/{basis,kernels,gates}.cpp,
src/photographiq/backends/native_fock.py and src/photographiq/gpu/fock.py):

1. Total-photon truncation basis, n_1 + ... + n_m < c, dimension C(m + c - 1, m), instead of a
   per-mode product basis of c^m states. This is a MODEL choice (the two truncations are different
   approximations), so the product basis is reported only as a dimension/memory count, never as
   an output-compared variant.
2. Fibre-local gate application (``apply_pure``): a k-mode gate acts on each fibre of fixed
   occupations of the other m - k modes through the leading block of its local matrix (P U P);
   no D x D operator is formed; cost sum_f dim_f^2. Compared, on the SAME P U P model, with
   (a) the global sparse D x D gate G assembled by ``photographiq.gpu.fock`` and applied with
   scipy.sparse (CPU) and cuSPARSE (GPU, CuPy), and (b) the dense D x D matrix of G (small D).
3. Photon-number conservation of passive gates: a passive Gaussian unitary (beam splitter,
   rotation) conserves the total photon number, so its local matrix is block diagonal in the
   sectors N = 0..c-1; ``gaussian_unitary`` skips the off-sector blocks (gates.cpp: "passive"
   branch). With total-photon truncation a passive gate is then exactly unitary on the truncated
   space (retained norm 1). The skip cannot be switched off from Python, so the block structure is
   verified (off-sector entries exactly 0) and its cost compared with a non-passive two-mode gate
   (CZ) of the same local dimension: an observation, not a controlled ablation.
4. Diagonal gates (rotation, Kerr, cross-Kerr) as phase multiplication (``phase_pure``) instead of
   a matrix application of the diagonal operator: same output, compared directly.
5. Caching: basis tables and the cubic-phase matrix (``functools.lru_cache`` in native_fock.py),
   and the sparse structure of the GPU/CPU gate (``_maps``/``_template`` in gpu/fock.py): first
   call vs cached call.

Accuracy: every variant's output is compared with the native fibre kernel (max |difference|,
states normalised). Memory: exact byte counts of the representation each variant needs
(``.nbytes`` of the arrays it holds: local matrix, CSR data+indices+indptr, dense matrix, state).
Timing: ``common.benchmark`` (warm-up excluded, median and IQR); final only with ``PGQ_IDLE_MACHINE=1``, otherwise provisional.
GPU timings include host<->device transfers (results returned to NumPy) and are taken only when
CuPy reports a usable device; ``Execution.device`` is recorded so a CPU fallback cannot be
mistaken for GPU execution (the GPU kernels never fall back).

In [ ]:
run_experiment("R46")

## R47: Native mixed-state (density-matrix) Fock backend validation.

Script: `15_native_fock/47_native_mixed_fock_validation.py`

Scientific question: does ``native-mixed-fock`` (compiled density-matrix engine, total-photon
truncation, P U P gates, Kraus thermal-loss channel, renormalisation with recorded retained
norm) produce physical density matrices that reproduce the exact mixed-state physics of
thermal/mixed Gaussian inputs, loss, partial traces and conditional measurements, and how do
the results converge with the cutoff?

References (independent of the engine):

* exact-in-truncation closed forms: thermal input rho_kk = nbar^k / (1 + nbar)^(k+1) restricted to
  k < c and renormalised; the reduced state of a two-mode squeezed vacuum built from two
  squeezers and a 50:50 splitter is diagonal with rho_kk ~ l^(2k), l = tanh r, for 2k < c (the
  truncated product state is P_tot of the exact state, which the passive splitter preserves);
  a displaced squeezed thermal input from dense expm on a 90-level space
  (``tests/validation/test_val_fock.py`` construction);
* channel invariants: pure loss on |alpha> gives the pure coherent state |sqrt(eta) alpha>
  (fidelity, purity), thermal loss gives <n> = eta |alpha|^2 + (1 - eta) n_th;
* Gaussian moments: the same Gaussian sequence (squeezers, splitter, loss, homodyne conditioning
  on a mixed two-mode state) on the NumPy ``gaussian`` backend; the Fock conditional state's
  quadrature mean and covariance (from exact ladder moments) and the outcome density must
  converge to the Gaussian ones;
* consistency: on a non-Gaussian sequence without loss, a pure input evolved by the mixed engine
  equals |psi><psi| of the pure ``native-fock`` engine;
* secondary: Piquasso 8.0.1 ``piquasso-mixed-fock`` on the same sequences at cutoff <= 8 only (its
  program validation dominated: 378 s for one cutoff-16 pass on this machine). CZ there is an
  Euler decomposition truncated after every factor, so it differs at truncation level.

Physicality after EVERY operation of the long sequence: Hermiticity residual, |tr - 1|, minimum
eigenvalue (``NativeMixedFockState.physicality``).

Assumptions/limitations: double precision; at most 3 modes (density matrices scale as
C(m + c - 1, m)^2); the photon-count probability check of the lossy TMSV uses the untruncated
reduced thermal state of the counted mode (analytic), so its deviation is truncation.

In [ ]:
run_experiment("R47")

## R48: Standard (Fock-state) boson sampling: permanents, distributions, samplers, scaling.

Script: `16_sampling/48_boson_sampling.py`

Scientific question: are PhotoGraphiQ's boson-sampling probabilities P(t|s) =
|Perm(U_{t,s})|^2 / (prod s! prod t!) (``photographiq.sampling``; native Gray-code Ryser kernel
and NumPy multiplicity-aware Ryser) correct for collision-free and collision (bunched) inputs and
outputs, normalised, consistent with a Fock-space simulation of the same interferometer, and are
the exact samplers distributed correctly? How do the permanent kernels scale?

References written or chosen to be independent of the code under test:

* permanents: the defining sum over permutations (brute force, n <= 8) and Glynn's formula
  (2^(n-1) Gray-free sum over sign vectors, n <= 16, numpy) implemented here;
* bunched permanents: brute force on the explicitly repeated matrix;
* distributions: an independent Fock-space evolution - the m-mode passive Gaussian unitary's Fock
  matrix from the native Bargmann/sector recurrence (``_native.fock.gaussian_unitary``; a different
  algorithm and code path from the permanent kernels) acting on |s> with cutoff n + 1; Piquasso
  8.0.1 PureFockSimulator with ``NumberState`` + ``Interferometer`` (same convention, checked);
* an exact physical law: the Fourier suppression law (Tichy et al. PRL 104, 220405): for the m x m
  DFT and the input with one photon in every mode, P(t) = 0 unless sum_k k t_k = 0 mod m;
* samplers: Pearson chi-square of 20000 seeded samples against the exact distribution.

Haar unitaries come from scipy.stats.unitary_group with fixed seeds (independent of
``photographiq.sampling.haar_unitary``). Timings are PROVISIONAL (shared machine).

In [ ]:
run_experiment("R48")

## R49: Gaussian boson sampling with photon-number-resolving (PNR) detectors.

Script: `16_sampling/49_gaussian_boson_sampling.py`

Scientific question: are PhotoGraphiQ's PNR probabilities P(n) = p_0 lhaf(A_{n+n}; gamma)/prod n!
(``photographiq.sampling.pnr_probability``: hafnians for undisplaced states, loop hafnians for
displaced ones, pure |lhaf(B_n)|^2 or mixed formula) correct for squeezed states through Haar
interferometers with loss and displacement, across modes, squeezing, photon number and cutoff?

References independent of the hafnian route:

* hafnian / loop hafnian: the defining sums over perfect matchings (with loops), written here,
  for 2N <= 12;
* Fock-space amplitudes: the product of single-mode squeezed vacua (native closed-form column
  ``gaussian_unitary(S)[:, 0]``) restricted to total photon number < c, then the m-mode passive
  interferometer's Fock matrix (native sector recurrence). For undisplaced, lossless inputs this is
  EXACT for every pattern with total < c (the truncated product is P_tot of the exact state and the
  passive unitary preserves sectors), so P(n) must agree to rounding at every cutoff. With
  displacement, P D P acts after the truncation, so agreement is a convergence in the cutoff;
* Piquasso 8.0.1 ``GaussianState.get_particle_detection_probability`` on the identical program
  (``benchmarks/gaussian_boson_sampling/bench_gbs._pq_state``), incl. loss and displacement;
* sum rule: the hafnian probability mass below the cutoff equals the exact retained mass of the
  Fock reference (undisplaced, lossless).

Haar unitaries from scipy.stats.unitary_group with fixed seeds.

In [ ]:
run_experiment("R49")

## R50: Threshold-detector GBS: torontonian click probabilities, cancellation and precision.

Script: `16_sampling/50_threshold_gbs.py`

Scientific question: how accurate are PhotoGraphiQ's threshold (click) probabilities
``photographiq.sampling.threshold_probability`` in double precision (native C++ and Python
sums), is the reported rounding bound B (``return_error`` / ``ThresholdInfo``) reliable, where
does the alternating inclusion-exclusion sum lose accuracy by cancellation (low probabilities,
many clicks, weak squeezing), and does the multiprecision path (``precision=<digits>`` and the
opt-in ``precision="auto"``) recover it? How does Piquasso 8.0.1 compare on the same states?

Method (REUSED, rerun here - no recorded numbers are presented as new):
``benchmarks/gaussian_boson_sampling/threshold_reference_matrix.py`` supplies the case builder
(``build``: Haar interferometer, squeezing r, transmissivity eta, optional displacement
alpha_j = 0.4 u e^{2 pi i v}, random k-click pattern; "zero" cases with an untouched vacuum mode
that clicks), the per-case evaluation ``run_case`` (native / Python double values with their
bound B, warnings, Piquasso on the identical program compared with a reference for ITS OWN stored
state) and the 40-digit mpmath reference ``reference`` (Schur complement on the dark modes, then a
depth-first Cholesky-extension walk over the 2^k subsets; separate code from PhotoGraphiQ's own
``precision=`` path; uncertainty 10^(3-40) sum|terms|). This script adds: the package's
``precision=50`` path and ``precision="auto"`` (path, digits, ThresholdInfo) per case, a
precision sweep (digits 16..60) on the hardest cases, the normalisation of full click
distributions (sum over 2^m patterns), and the acceptance tests below. The earlier recorded matrix
(docs/validation/gbs-threshold-accuracy.md, results/threshold_reference_matrix_20261111-233720.json)
is cited for comparison only.

Do NOT read double-precision results as "full double precision": the relative error grows with the
cancellation ratio sum|terms| / |P| (reported per case) and exceeds 1 for many-click, weak-squeezing
cases; the bound B is what the function promises, and ``precision="auto"`` is the remedy.

In [ ]:
run_experiment("R50")

## R51: CV flow finding on weighted graphs, checked against the defining linear algebra.

Script: `17_flow_calculus/51_cv_flow_finding.py`

Scientific question: does ``photographiq.flows.find_cv_flow`` (backward real-linear layering,
Booth and Markham, Quantum 7, 1146 (2023), Defs. 5-6 / Lemma 7) decide CV-flow existence
correctly on linear wires, 2D clusters, branched graphs, weighted random graphs and graphs
without a supported certificate, and are its layers, correction coefficients and
impossibility witnesses mathematically valid?

Flow condition (real, weighted; A the weighted adjacency): a total order of the measured
nodes admits CV flow iff for every measured u there is a real vector c_u supported on the
non-input nodes measured after u (or outputs) with (A c_u)_v = delta_uv for every v measured
up to and including u. Corrections: q_t -= c_t m, p_t -= (A c)_t m, m = record / sin(theta).

Independent references computed in this script (no photographiq flow code):

1. Exact forward search: rational arithmetic (``fractions.Fraction`` of the float64 weights)
   over prefix SETS (the condition for u only depends on the set measured up to u), with
   e_u in range(M) decided by exact rank(M) == rank([M | e_u]). Complete for existence; it
   explores the subset lattice forward, whereas the library layers backwards from outputs.
2. Least-squares re-derivation on the reported total order: c = lstsq(M, e_u), residual
   ||M c - e_u||_inf and the rank gap rank([M|e_u]) - rank(M) by SVD.
3. Layered-certificate check: the library's layer vectors c_u are tested for support (later
   layers / outputs, never inputs) and residual on u's own and earlier layers; the reported
   correction_map must equal (q, p) = (-c_t, -(A c)_t).
4. Impossibility witnesses: y^T A[W, C] = 0 exactly with y_u != 0 (W unsolved nodes,
   C the solved non-input nodes) is re-checked in rational arithmetic.

Backend: pure linear algebra plus the library's float64 Gaussian determinism check (Gaussian
backend, exact postselection) for certified graphs; resource squeezing r = 1.0, unit-
efficiency homodyne. Seed 20261151 for the random graph family.

Limitations: the brute-force reference is exponential in the number of measured nodes
(used up to 16); float64 weights are taken as exact rationals; "impossible" means no
admissible total order for the real-linear CV-flow condition, never a proof of
nondeterminism by other means; finite squeezing is never called an ideal unitary.

In [ ]:
run_experiment("R51")

## R52: CV-MBQC determinism: outcome independence in the ideal limit, finite-squeezing noise apart.

Script: `17_flow_calculus/52_cv_mbqc_determinism.py`

Scientific question: for CV-flow corrected Gaussian patterns, (i) does the outcome-averaged
channel (S, N) equal the intended map S at every finite squeezing r, with the added noise N
reported separately and vanishing as r -> oo; (ii) is the corrected conditional output
mean independent of the homodyne records in the ideal limit (residual dependence G ~ e^{-2r}),
with outcome-independent conditional covariance; and (iii) do the library's channel and
postselected trajectories agree with an independent model?

Cases: single-mode teleportation (one wire step), a weighted 4-node wire, a 4-mode graph
teleportation (inputs R_i, cycle cluster C_i), a weighted 2x3 cluster, and a branched graph
(1 input -> 2 outputs, an ideal isometry with one infinitely squeezed output quadrature).
Corrections come from ``photographiq.flows.find_cv_flow``; the executed pattern is
``photographiq.flows.cv.corrected_pattern`` built from exactly that correction map.

Independent reference (this script, no library simulation code): linear Wigner-variable
model. Variables x = (input quadratures, (q_j, p_j) per resource node) with covariance
diag(V_in, e^{2r}, e^{-2r}, ...); CZ(w): p_u += w q_v; records m_u = k_u q_u + p_u
(= record / sin(theta), theta = atan2(1, k_u)); corrections q_t += c_q m_u, p_t += c_p m_u (correction_map convention).
Output rows L, record rows R give S = L_in, N = L_res Sigma L_res^T = e^{2r} N_+ + e^{-2r} N_-
(exact split by the anti-squeezed / squeezed resource columns), and Gaussian conditioning
E[out | m] = L mu + G (m - R mu), G = L Sigma R^T (R Sigma R^T)^{-1}. Closed forms for S:
T(k) = [[-k, -1], [1, 0]] per unit step, [[-k/w, -1/w], [w, 0]] per weighted step, and
C_Z(A) (+)_i [[-s_i, -1], [1, 0]] for graph teleportation.

Library quantities compared: ``gaussian_channel`` (S, N) and ``simulate(backend='gaussian',
measurement_outcomes=...)`` (exact postselection: each run is ONE conditional branch, never a
deterministic execution) plus seeded sampled trajectories, and ``verify_cv_determinism``.

Finite squeezing: S is r-independent, but N != 0 and G != 0 at every finite r; no finite-r
channel is called an ideal unitary. Outcome dependence is measured on the finite subspace
F = null(N_+) (the ideal-limit squeezed output directions of an isometry are excluded).
Seed 20261152. Unit-efficiency homodyne, lossless resources.

In [ ]:
run_experiment("R52")

## R53: Qubit causal flow, gflow and Pauli flow against Graphix 0.4 and the defining conditions.

Script: `17_flow_calculus/53_qubit_flows_graphix.py`

Scientific question: do ``photographiq.flows.find_causal_flow`` / ``find_gflow`` /
``find_pauli_flow`` agree with Graphix 0.4 on flow existence where the two use the same
semantics (qubit open graphs, planes XY/XZ/YZ, Pauli labels X/Y/Z, identical observable
conventions: XY cos X + sin Y, XZ cos Z + sin X, YZ cos Z + sin Y), and do the returned
flows -- ours AND Graphix's -- satisfy the defining conditions and give strongly
deterministic patterns?

Definitions checked by an independent checker written in this script (plain set algebra;
the library verifiers are not used), with the strict order u < v iff layer(u) < layer(v) and
outputs after every layer:

* causal flow (Danos-Kashefi 2006): f(i) ~ i, i < f(i), every neighbour k != i of f(i) has
  i < k; f injective into non-inputs; XY measurements only;
* gflow (Browne et al. 2007, three-plane form of Backens et al. 2021): G1 j in g(i)\{i} =>
  i < j; G2 j in Odd(g(i))\{i} => i < j; G3 XY: i notin g(i), i in Odd; XZ: i in g, i in Odd;
  YZ: i in g, i notin Odd;
* Pauli flow (Simmons 2021, Def. 2.4): P1-P3 relaxations for X/Y/Z-labelled nodes, P4-P9
  self conditions.

Determinism: an independent dense statevector simulator (Choi vector with one reference qubit
per input; projective measurement on the eigenvector of the measured observable; X/Z
corrections X_{p(u)\{u}}, Z_{Odd(p(u))\{u}} on nodes still alive) checks that every one of
the 2^m branches has probability 2^-m and the same output state up to phase.

Graphs: textbook successes (line, 2x3 cluster), gflow without causal flow, YZ/XZ planes,
Pauli flow needed (X measurements labelled in the XZ plane), Y/Z Pauli measurements, and
failures (|I| > |O|, XZ node inside an input line, XZ labels without Pauli relaxation),
plus seeded random open graphs (n = 3..8). CV graphs are never given qubit certificates:
``analyze(cv_graph, 'gflow')`` must be ``unsupported``.

Limitations: Graphix's correction sets need not equal ours (flows are not unique), so sets are
compared only as "both valid"; depth is compared (ours is maximally delayed). Statevector
determinism is limited to <= 12 simulated qubits. Seed 20261153.

In [ ]:
run_experiment("R53")

## R54: Qubit measurement calculus (DKP) with full branch semantics; CV command view only.

Script: `17_flow_calculus/54_measurement_calculus.py`

Scientific question: do the measurement-calculus rewrites of ``photographiq.calculus.qubit``
(standardization to NEMC form, signal shifting) preserve the full process semantics, branch
by branch including every measurement outcome, and are adaptive measurements, corrections and
classical dependencies tracked correctly? For CV patterns: are commands classified correctly
and displacement corrections extracted as exact affine forms of the records? (No complete CV
measurement calculus is implemented or claimed.)

Qubit semantics (DKP, Danos-Kashefi-Panangaden 2007): N |+>, E CZ, M^{s,t}(plane, a) measures
the observable Z^t X^s O(plane, a) X^s Z^t with s, t the parities of its domains (outcome 0 =
+1 eigenvalue; XY: cos X + sin Y, XZ: cos Z + sin X, YZ: cos Z + sin Y), X/Z apply iff
their domain parity is 1, S flips a stored signal. Each branch is keyed by the FINAL signals of
the measured qubits (signal shifting relabels raw outcomes and restores the signals with S).

References computed in this script:

1. Independent dense simulator: Choi vector of each branch (one reference qubit per input in a
   Bell pair), measuring the conjugated observable built as a matrix (not via an angle table)
   and projecting on its (-1)^r eigenvector. Branches compared up to a branch phase (the
   eigenvector phase is a convention); probabilities ||v_b||^2 compared without any freedom.
2. Graphix 0.4 statevector simulator with ``FixedBranchSelector`` on the original patterns
   (angles in units of pi; same plane conventions), random normalized input state.
3. Library ``run_branch`` operators, compared with (1).

Patterns: seeded random valid non-standard patterns (interleaved N/E/M/X/Z, random planes,
angles and s/t domains: adaptive measurements and corrections), and flow-derived standard
patterns (``pattern_from_flow`` from causal flow / gflow / Pauli flow certificates).

CV part: ``photographiq.calculus.cv.command_view`` on CV-flow corrected patterns: role
classification counts against the command list, and every affine correction form evaluated on
random records against direct evaluation of the original expression; a deliberately nonlinear
feed-forward must be reported as non-affine.

Seed 20261154. Limitations: dense simulation (<= 12 simulated qubits); random patterns have 2
inputs, 3 ancillas, 2 outputs; Graphix comparison only where its commands coincide (no S).

In [ ]:
run_experiment("R54")

## R55: Local complementation and local-Clifford equivalence of graph states (qubit), and the
validated Gaussian graph rules (CV).

Script: `17_flow_calculus/55_local_clifford.py`

Scientific question: (i) does ``photographiq.calculus.clifford.local_complement`` with its local
Clifford ``U_v = exp(-i pi/4 X_v) prod_{w in N(v)} exp(+i pi/4 Z_w)`` map |G> to |tau_v(G)> up to a
global phase (Van den Nest, Dehaene, De Moor, PRA 69, 022316 (2004)); (ii) do LC sequences
found by ``lc_equivalence`` produce Clifford witnesses that are correct on statevectors, and are
negative answers consistent with an LU invariant; (iii) for CV, are the implemented
``GaussianGraph`` rules (Menicucci-Flammia-van Loock 2011: CZ, shear, squeezing, rotation, beam
splitter, ideal q/p homodyne) consistent with phase-space evolution of the covariance?

Conventions: |G> = prod_{(u,v) in E} CZ_uv |+>^n, big-endian in the stated node order; states
compared up to a global phase, which is reported (arg <tau_v G| U_v |G>); Clifford matrices are
the library's canonical representatives (phase fixed), so composed witnesses also hold only up
to a global phase. hbar = 2, interleaved (q, p), vacuum covariance I for the CV part.

Independent references in this script: graph states built by applying CZ phases to |+>^n;
local complementation by toggling neighbourhood edges; U_v built from cos/sin of Pauli matrices;
LU invariant: the vector of Schmidt ranks over all bipartitions (a labelled-LC invariant: if it
differs, no LC sequence exists). CV: covariance propagated as S V S^T with the gates' symplectic
matrices, and ideal homodyne by Gaussian conditioning (Schur complement of the covariance in the
infinite-precision limit), compared with ``GaussianGraph.to_covariance()`` and its mean.

Not covered: CV local complementation is NOT implemented (``cv_local_complementation`` raises);
no claim about LC orbits of CV graphs. Seed 20261155.

In [ ]:
run_experiment("R55")

## R56: Qubit ZX-calculus subset: rewrite soundness with exact scalars (QUBIT ZX ONLY; no CV-ZX).

Script: `17_flow_calculus/56_zx_calculus.py`

Scientific question: does every implemented rewrite of ``photographiq.calculus.zx`` (spider
fusion, identity removal incl. Hadamard-Hadamard cancellation, colour change, Pauli state copy,
2x2 bialgebra, Hopf, parallel Hadamard-edge cancellation, self-loop removal, and the
``simplify_spiders`` fixed point) preserve the linear map of the diagram EXACTLY, including
its scalar, and do graph-state and MBQC-branch diagrams have the intended semantics?

Semantics (Coecke-Kissinger, exact scalars): Z spider with phase a and n legs is
|0..0><0..0| + e^{ia} |1..1><1..1|; X spider is |+..+><+..+| + e^{ia} |-..-><-..-|; simple edge
= identity, Hadamard edge = H; the diagram's matrix is 2^|out| x 2^|in| (first boundary most
significant) times the diagram's stored scalar.

Independent reference (this script): an own tensor-network evaluator that reads only the
diagram's vertex/edge/boundary data and scalar, builds Z spiders from computational-basis
projectors and X spiders DIRECTLY from |+>/|-> outer products (not as H-conjugated Z spiders),
and contracts with ``numpy.einsum``. Closed-form matrices for the textbook examples and
statevectors built from CZ phases for graph states; ``qubit.run_branch`` Kraus operators for
the MBQC branch diagrams.

Scalar / phase convention: rewrites carry exact scalar factors (e.g. Hopf 1/2, bialgebra
sqrt 2, copy 2^{(1-n)/2} e^{i a alpha}); before/after matrices are compared with NO global
phase or normalisation freedom. Only for diagrams with an unrecorded convention (none here)
would a phase be divided out. The proportionality constant after/before is reported (must be 1).

Scope: qubit ZX subset only. Pi-copy, local complementation/pivoting as ZX rewrites, and any
CV-ZX identities are not implemented and not claimed. Dense evaluation (<= ~14 open + internal
legs per contraction step). Seed 20261156.

In [ ]:
run_experiment("R56")

## R57: Interactive visualization validation (one validated graph, four views).

Script: `18_visualization/57_interactive_visualization.py`

Builds the complete-graph, flow, correction and execution views of ONE validated
graph (the compiled two-mode Gaussian circuit R(0.4) (+) S(0.3), then CZ(0.8), at
resource squeezing r = 1.5) and checks programmatically that

1. changing visual positions (every offered layout plus random manual layouts) leaves the
   OpenGraph, the pattern, the drawn flow/correction data and a seeded simulation
   bit-for-bit unchanged;
2. the drawn flow arrows equal ``flows.analyze(...).correction_map`` and the drawn
   correction arrows/labels equal the outcome-dependent ``Displace`` commands of the
   pattern, read here directly from the command expressions (independent of
   ``pattern_corrections``);
3. exported static SVG files match the model (node/edge/arrow groups and labels parsed
   from the SVG), and every routed edge keeps the documented clearance from other nodes;
4. the browser-format export (graph + layout JSON), layout files and OpenGraph JSON round
   trip restore positions and metadata exactly;
5. in a real browser (Playwright, installed Edge; real mouse/select/click events) a node
   drag, a stage switch and a layout switch keep the exported graph identical and the
   drawn arrows equal to the model; the full ``scripts/check_interactive_browser.py``
   suite runs as well;
6. the view renders as a notebook output (nbclient executes a notebook; the iframe
   ``srcdoc`` it produced is then opened in the browser).

Steps 5-6 need Playwright (``PYTHONPATH`` pointing at a venv that has it) and a local
Cytoscape.js copy in ``PHOTOGRAPHIQ_CYTOSCAPE_JS`` (or network access to the CDN). When
they cannot run they are recorded as ``not_run`` (never as passing); ``--require-browser``
turns that into a failure.

Usage: python 57_interactive_visualization.py [--require-browser]

In [ ]:
run_experiment("R57")

## R58: Batched Gaussian kernels: NumPy CPU vs native C++ CPU vs CuPy GPU (accuracy and stage timing).

Script: `19_gpu/58_gpu_gaussian.py`

Scientific question: for batches of B independent n-mode Gaussian states, how do the NumPy
(vectorised CPU), native C++/OpenMP batch kernels and the CuPy GPU path of
``photographiq.gpu`` compare in (i) accuracy against the float64 NumPy reference and (ii) time,
with the GPU time SEPARATED into host data preparation, host->device transfer, kernel
execution and device->host transfer? Did the GPU actually execute (no CPU fallback)?

Operations (identical inputs for all engines; hbar = 2, interleaved, vacuum V = I):

* ``local_symplectic``: beam splitter (theta = 0.37) on modes (n/3, 2n/3), O(B n) work;
* ``dense_symplectic``: a different random 2n x 2n symplectic per state on all modes (batched
  GEMM, O(B n^3)); S_b = O(U_b) D(r_b) with U_b Haar-random unitary (QR of complex Gaussian) and
  single-mode squeezers |r| <= 0.3;
* ``homodyne``: conditioning on a given outcome 0.1 at angle 0.4, efficiency 0.9, mode n/3
  (Schur complement, O(B n^2)); returns conditioned moments and densities (postselected
  branches, not sampled trajectories);
* ``purity``: det(V)^(-1/2) by batched slogdet (no native batch kernel: NumPy vs GPU only).

Input: B copies of a physical state (squeezed CZ chain with thermal loss, ``GaussianBackend``),
so every engine sees the same data. Timing: ``time.perf_counter`` on the host for CPU engines and
GPU end-to-end; CUDA events (``cupy.cuda.Event``, synchronised) for the GPU stages H2D, kernel
(device-resident input, ``on_device=True``; includes the library's validity-check syncs) and
D2H. Median of ``--repeats`` after ``--warmup`` calls; first calls (CuPy compilation, handle
creation) are never samples.

Device verification: every GPU result must report ``execution.device == 'cuda:<id>'``,
backend 'cupy', and device-resident outputs must be device memory
(``pointerGetAttributes(ptr).type == memoryTypeDevice``); otherwise the script fails. No CPU
number is ever labelled GPU. Precision: float64 everywhere; float32 GPU rows only as an
explicit, separately labelled accuracy/speed trade-off. CUDA-Q is not used.

Timings are final only when the run declares an idle machine (``PGQ_IDLE_MACHINE=1``);
the manuscript configuration is ``--full``. Seed 20261158.

In [ ]:
run_experiment("R58")

## R59: PhotoGraphiQML physical execution on the unified native backend.

Script: `20_photographiqml/59_photographiqml_physical.py`

Scientific question: when the restricted (signed-X, angles 0/pi) PhotoGraphiQML MuTA is
lowered to finite-energy GKP modes and executed on PhotoGraphiQ's compiled ``native-fock``
engine, does it reproduce the Piquasso-backed ``piquasso-fock`` execution (decoded outcome
probabilities, conditional final states, measurement records, sampled shots), converge in
the Fock cutoff, and keep the logical-to-physical interface (lowering, Pauli-frame
corrections, decoded logical observables) consistent with the ideal logical MuTA?

Model and conventions: hbar = 2; GKP codewords are finite-energy (Gaussian peaks of width
``peak_width`` under a Gaussian envelope ``envelope``, ``peaks`` peaks), projected onto a
TOTAL-photon-number Fock cutoff. Logical nodes are GKP modes, CZ is the unit-gain
``Entangle`` (p_u += q_v, p_v += q_u), each measured node is a p-homodyne decoded by the
nearest-cell GKP decoder (lattice spacing sqrt(2 pi)); decoded bits drive a classical
logical Pauli frame (``photographiqml.lowering.node_frame``) that is applied to the final
joint X/Z modular readout. Finite GKP codewords are NOT orthogonal (Gram overlap reported),
so the physical instrument is not the ideal logical unitary: the logical-physical gap is a
property of the finite resource, reported separately from engine/truncation error.

Sections (all quantities are saved; the figure shows the first five):

A. Engine agreement (physical-conditional, fixed analog outcomes): native-fock vs
   piquasso-fock at the same cutoff -> max |dP| of decoded joint probabilities, final-state
   infidelity 1 - F (squared Uhlmann), identical decoded bits, retained Fock norm.
B. Cutoff convergence: decoded probabilities at increasing total-photon cutoff, measured
   against the largest native cutoff (same resource); Piquasso is measured against the same
   native reference, so both engines are shown to approach one limit.
C. Pauli-frame corrections on physical data: all 2^4 lattice-point branches (analog
   outcomes 0 or sqrt(2 pi) per measured node) of a 1-wire MuTA, final readout with and
   without the decoded frame, at three GKP resource levels. Each branch is a POSTSELECTED
   conditional trajectory, not a deterministic outcome.
D. Shot statistics (physical-shots): identical seeds on both engines -> sampled analog
   outcomes, decoded bits and sampled output bits; Rao-Blackwell (mean conditional POVM)
   probabilities with their standard errors; empirical frequencies with binomial SEs.
E. Runtime (provisional, shared machine) and memory (Hilbert dimension, dense state bytes,
   tracemalloc peak of NumPy-visible allocations; native C++ allocations are NOT visible to
   tracemalloc).
F. Interface checks: logical target from the physical result equals the logical
   ``MuTA.unitary`` applied to the input (independent code path, validated against MentPy
   and analytic Euler/XX identities in PGML-R2/R3/R9); unsupported angles (pi/2) are
   rejected before any Fock allocation on both backends; ``native-mixed-fock`` is rejected.

Independent references: Piquasso 8.0.1 (independent Fock engine; the finite-GKP codeword
amplitudes and lowered Pattern are SHARED inputs, so engine agreement is independence
class C in the PGML taxonomy), logical MuTA (ideal qubit reference).

Cross-references (not duplicated here): PGML-R34 GKP codeword projection, PGML-R35/R36
capability map and lowering preservation, PGML-R37 exhaustive logical Pauli-frame
branches, PGML-R38/R41 independently assembled Pattern comparison (piquasso-fock),
PGML-R43/R56 decoded logical-physical gap, PGML-R44 shot convergence, PGML-R47/R58
resource-axis convergence.

Limitations: no loss or detector inefficiency, no arbitrary-angle physical measurement, no
fault-tolerance or threshold claim, no decoded-state fidelity to a logical qubit state
(only decoded outcome statistics are compared); timings are provisional (shared machine).

In [ ]:
run_experiment("R59")

## R60: End-to-end showcase, from a target Gaussian operation to validated output states.

Script: `21_showcase/60_end_to_end_showcase.py`

Pipeline (every number below comes from a real library call or an independent oracle):

1. target: S = R_1(-0.6) CZ(0.8) [R_0(0.4) (+) S_1(0.3)] on two modes (hbar = 2,
   interleaved (q0, p0, q1, p1), R(t) = [[cos t, -sin t], [sin t, cos t]],
   S(r) = diag(e^-r, e^r), CZ(g): p0 += g q1, p1 += g q0);
2. circuit ``pg.Circuit`` -> MBQC pattern ``Circuit.compile(squeezing=r)``;
3. complete weighted cluster graph ``GraphView.from_pattern`` (roles, weights, squeezing,
   homodyne shears) before any flow analysis;
4. flow/determinism analysis ``flows.analyze`` (certified CV flow, layers, correction map,
   ideal-limit determinism map);
5. measurement schedules: the flow's measurement order respects every dependency of its
   correction map, and the compiled command order respects every dependency of the
   feed-forward it executes (the two schedules differ; both are reported);
6. classical correction map: the flow's map (standard form) and the compiled feed-forward
   (``Displace`` commands) are both drawn; a pattern executing exactly the flow's map
   (``flows.cv.corrected_pattern``) is shown to give the same channel as the compiled one;
7. ``native-gaussian`` simulation of K trajectories (distinct seeds, coherent inputs);
8. corrected output-state analysis: per-trajectory conditional covariance (outcome
   independent), conditional means scatter, ensemble mean/covariance vs S mu, S V S^T + N;
9. finite-squeezing noise: intended map S (r independent) and added noise N(r); N(r) e^{2r}
   is checked to be constant (every resource contributes variance e^{-2r});
10. interactive page (all four stages) and static exports;
11. independent validation: analytic symplectic product (NumPy, written here) and the
    NumPy ``gaussian`` backend (same seeds, same trajectories);
12. runtime and reproducibility metadata (timings are provisional on a shared machine).

Usage: python 60_end_to_end_showcase.py [--quick | --full] [--repeats N]

In [ ]:
run_experiment("R60")

## Summary and manuscript tables

In [ ]:
print(json.dumps(RECORD, indent=1))
import generate_tables  # noqa: E402

generate_tables.main()
